In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ (her notebook'un ilk hücresi) ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "a/1.9-split"                                  # merge edildikten sonra "main"
try:
    TOKEN = userdata.get("GH_TOKEN")                        # sadece private repo için gerekir
except userdata.SecretNotFoundError:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .
if "/content/repo/src" not in sys.path:      # editable install is only seen after a kernel restart
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()          # config/config.yaml + DATA_ROOT
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

# 09 — Splits, preprocessing pipeline and class imbalance

Stage 1.9. This notebook contains no business logic; it only calls `recsys` modules.

* Input: stage **`loaded`** (LATEST), validated against its schemas.
* Output: ONE atomic **`splits`** version with both splits, the targets, the fitted
  preprocessing pipeline, the encoded catalogue and the development sample (each table is
  validated against `SPLITS_OUTPUT_SCHEMA`).
* RAW_recipes is never split (it is the catalogue); RAW_interactions is split two ways.
* Every `fit` below uses TRAIN data only. The one exception is the stratification label, which is
  computed from all interactions and used only to stratify the grouped split.

In [ ]:
from recsys import registry
from recsys.contracts import (
    LOADED_INTERACTIONS_SCHEMA,
    LOADED_RECIPES_SCHEMA,
    RECIPE_FEATURES_SCHEMA,
    SPLITS_OUTPUT_SCHEMA,
    assert_class_proportions,
    assert_disjoint,
    assert_rows_preserved,
    assert_temporal_order,
    validate,
)
from recsys.data.split import (
    assignments_table,
    class_balance_table,
    make_dev_sample,
    split_summary,
)
from recsys.imbalance.resampling import class_weights, resampling_summary
from recsys.io import ArtifactStore
from recsys.pipeline.build import PreprocessingPipeline, build_step

SPLIT_CFG = CFG.section("splits")
PIPE_CFG = CFG.section("pipeline")
RESAMPLING = CFG.section("resampling")["strategy"]

store = ArtifactStore.from_config(CFG)
loaded = store.load("loaded")
recipes = validate(loaded["recipes"], LOADED_RECIPES_SCHEMA)
interactions = validate(loaded["interactions"], LOADED_INTERACTIONS_SCHEMA).reset_index(drop=True)
print("input: loaded", loaded.version, "| recipes", recipes.shape, "| interactions", interactions.shape)

## 1. Two splits

* **(a) grouped, stratified** (`group_stratified`): whole recipes go to train / val / test
  (70 / 15 / 15 of the recipes), stratified on the recipe rating class (`unrated` for recipes
  without any rating > 0). For the recipe-level rating models.
* **(b) temporal per user** (`temporal_user`): for users with at least `min_interactions`
  interactions the latest one is test and the one before it validation. For personalisation.

In [ ]:
label_binner = build_step(PIPE_CFG["target"]["params"]["binner"], CFG.raw)
grouped_splitter = registry.create(
    "splitter", "group_stratified",
    label_binner=label_binner,
    random_state=CFG.random_state,
    unrated_label=SPLIT_CFG["stratify_unrated_label"],
    **CFG.split,
)
temporal_splitter = registry.create("splitter", "temporal_user", **SPLIT_CFG["temporal"])
grouped = grouped_splitter.split(interactions, recipes)
temporal = temporal_splitter.split(interactions, recipes)
labels = grouped_splitter.labels_

# ACID checks: nothing is written unless all of them pass
for res in (grouped, temporal):
    assert_rows_preserved(interactions, res.as_dict())
assert_disjoint(grouped.as_dict(), "recipe_id")
assert_class_proportions(labels, tolerance=SPLIT_CFG["class_share_tolerance"])
assert_temporal_order(temporal.as_dict())
print("checks passed: rows preserved, recipe_id disjoint, class shares, temporal order")

summary = split_summary({"group_stratified": grouped, "temporal_user": temporal})
balance = class_balance_table(labels)
display(summary)
balance.pivot(index="label", columns="split", values="share")

## 2. Targets and preprocessing pipeline (fit on grouped TRAIN only)

`PreprocessingPipeline` is built from the `pipeline` block of `config/config.yaml` through the
registry: missing values (1.3) -> outliers (1.4) -> binning (1.7) -> log1p / scaling (1.4 / 1.8)
plus the ingredient normaliser, rare filter and encoder (1.5). Steps marked `optional` that are
not registered yet are listed under `skipped`. Val / test targets use their own interactions but
the TRAIN class boundaries.

In [ ]:
pipeline = PreprocessingPipeline.from_config(CFG).fit(recipes, grouped.train)
display(pipeline.step_labels())

features = validate(pipeline.transform(recipes), RECIPE_FEATURES_SCHEMA)  # whole catalogue
ingredient_matrix = pipeline.encode(features)
assert len(pipeline.vocab_) == ingredient_matrix.shape[1]
targets = {name: pipeline.targets(frame) for name, frame in grouped.as_dict().items()}
print("features", features.shape, "| ingredient matrix", ingredient_matrix.shape,
      "| fitted on", pipeline.n_fit_recipes_, "train recipes")
{name: t["rating_class"].value_counts().to_dict() for name, t in targets.items()}

## 3. Class imbalance (TRAIN only)

Class weights are always reported; random under / over sampling are shown for comparison and
would only ever be applied to the train set (`resample_train`). Validation and test keep the
real distribution.

In [ ]:
y_train = targets["train"]["rating_class"]
weights = class_weights(y_train)
resampling = resampling_summary(
    y_train,
    {
        "random_under": registry.create("resampler", "random_under", random_state=CFG.random_state),
        "random_over": registry.create("resampler", "random_over", random_state=CFG.random_state),
    },
)
print("configured resampling:", RESAMPLING, "| class weights:", weights)
resampling.pivot(index="label", columns="method", values="n_rows")

## 4. Development sample

A stratified sample of recipes (on the same label) with all their interactions, for fast
experiments. Because the grouped split assigns whole recipes, `grouped_assignments` restricted
to these recipes is a valid split of the sample.

In [ ]:
dev_recipes, dev_interactions = make_dev_sample(
    recipes, interactions, labels, SPLIT_CFG["dev_sample"]["n_recipes"], CFG.random_state
)
dev_share = dev_recipes["label"].value_counts(normalize=True)
all_share = labels["label"].value_counts(normalize=True)
print("dev recipes", len(dev_recipes), "| dev interactions", len(dev_interactions))
(dev_share - all_share).abs().max()

## 5. Save stage `splits` (one atomic version)

In [ ]:
splits = store.save(
    "splits",
    {
        "grouped_assignments": assignments_table(grouped),
        "temporal_assignments": assignments_table(temporal),
        "stratification_labels": labels,
        "targets_train": targets["train"],
        "targets_val": targets["val"],
        "targets_test": targets["test"],
        "split_summary": summary,
        "class_balance": balance,
        "dev_recipes": dev_recipes[["id", "label"]],
        "resampling_summary": resampling,
        "recipe_features": features,
        "ingredient_matrix": ingredient_matrix,
        "vocab": {"vocab": pipeline.vocab_},
        "class_weights": weights,
        "pipeline": pipeline,
        "pipeline_state": pipeline.fitted_state(),
    },
    inputs=[loaded],
    params={
        "split": dict(CFG.split),
        "splits_config": SPLIT_CFG,
        "pipeline_config": PIPE_CFG,
        "pipeline_steps": pipeline.step_labels(),
        "grouped_params": dict(grouped.params),
        "temporal_params": dict(temporal.params),
        "resampling": RESAMPLING,
        "random_state": CFG.random_state,
    },
    schemas=SPLITS_OUTPUT_SCHEMA,
)
print("saved:", splits.stage, splits.version, "->", splits.path)
check = store.load("splits")  # re-read with sha256 verification
{name: f.get("n_rows", f["kind"]) for name, f in check.manifest["files"].items()}